# Task 1 — Repository discovery

Recorded run against the pinned LeRobot checkout on 2026-07-25. The build reads
the saved run summary, so the upstream repository cannot change the report.


In [1]:
from pathlib import Path
import json
from IPython.display import display

def find_evidence():
    start = Path.cwd().resolve()
    for root in (start, *start.parents):
        candidate = root / 'docs' / 'evidence' / 'live_pipeline_summary.json'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('docs/evidence/live_pipeline_summary.json not found')

evidence_path = find_evidence()
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
task = evidence['task1']
display({
    'captured_at': evidence['captured_at'],
    'repository': task['repository'],
    'shallow_clone': task['shallow_clone'],
    'commit': task['commit'],
    'raw_python_files': task['raw_python_files'],
    'included_python_files': task['included_python_files'],
    'excluded_python_files': task['excluded_python_files'],
    'first_manifest_entry': task['first_manifest_entry'],
})

{'captured_at': '2026-07-25T03:30:43.726347Z',
 'repository': 'https://github.com/huggingface/lerobot.git',
 'shallow_clone': True,
 'commit': '0d383d09f2051444de211739196a28cc94736861',
 'raw_python_files': 752,
 'included_python_files': 490,
 'excluded_python_files': 262,
 'first_manifest_entry': {'relative_path': 'scripts/ci/extract_task_descriptions.py',
  'file_size_bytes': 8631,
  'file_hash': '7d1235a0b11643c68de0b5ae6de60c71c999f08b575ac07c91848abb440f40cb'}}

In [2]:
assert task['repository'] == 'https://github.com/huggingface/lerobot.git'
assert task['shallow_clone'] is True
assert len(task['commit']) == 40
assert task['raw_python_files'] == 752
assert task['included_python_files'] == 490
assert task['excluded_python_files'] == 262
assert task['raw_python_files'] == task['included_python_files'] + task['excluded_python_files']
assert task['duplicate_relative_paths'] == 0
assert task['excluded_pattern_paths_remaining'] == 0
assert len(task['first_manifest_entry']['file_hash']) == 64
result = {
    'status': 'PASS',
    'shallow_clone_verified': True,
    'manifest_files': task['included_python_files'],
    'duplicate_relative_paths': 0,
    'excluded_pattern_paths_remaining': 0,
}
display(result)

{'status': 'PASS',
 'shallow_clone_verified': True,
 'manifest_files': 490,
 'duplicate_relative_paths': 0,
 'excluded_pattern_paths_remaining': 0}

## Reflection

Pinning the shallow checkout turns discovery counts into reproducible evidence. Filtering both path components and filenames avoids test/setup/generated leakage, while sorted relative paths and SHA-256 hashes make exact reruns comparable.